In [1]:
from fretboardnonredundant import get_butterworth_group_delay,num_harmonics

print(get_butterworth_group_delay(40,1,6,48000))

556


In [2]:
import jams
import numpy as np
from pathlib import Path
from common import OUTPUT_DIM_NOTES,Q_FACTOR
#pip install pyguitarpro
import guitarpro
from joblib import Parallel, delayed
import math
import time
import glob
import os
from fretboardnonredundant import get_butterworth_group_delay,num_harmonics
# Define the directory containing JAMS files. The corresponding input audio files have identical names but with .wav extension. These names are deduced
# from the jams files automatically.
# jams_dirs=['/home/gerald/guitarset/annotation/rock']
# Configuration

limit_gb = 500
limit_bytes = limit_gb * (1024**3)

jams_root_dir='/data2/converted_audio_jams'

# jams_files=glob.glob(os.path.join(jams_root_dir,'**','*.jams'),recursive=True)

# #  Get the parent directories of all jams files
# jams_dirs=set()
# for file in jams_files:
#     parent_dir=os.path.dirname(file)
#     jams_dirs.add(parent_dir)



output_data_dir='/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices'
output_data_dir='/data2/data_slices'
chunk_size=1 

def convert_period_s_to_samples(start_s,end_s,sample_rate):
    start_frame=int(math.floor(start_s*sample_rate))
    end_frame=int(math.ceil(end_s*sample_rate))
    return start_frame,end_frame

def load_jams_file(file_path):
    with open(file_path, 'r') as f:
        jams_data = jams.load(f,validate=False)
    return jams_data
# intervals += (60 / tempo) * num_ticks / guitarpro.Duration.quarterTime
#Extract from synthtab jams file the note_tab annotations
def extract_guitar_annotations(jams_data,duration_bin_s=1000):
    #Get the tempi from the jams file
    tempos=[]
    for annotation in jams_data.annotations:
        if annotation.namespace=='tempo':
            for obs in annotation.data:
                tempos.append((obs.time,obs.value)) #(time,bpm)

    # Sort the tempos by ticks
    tempos=sorted(tempos,key=lambda x:x[0])

    has_multiple_tempos=len(tempos)>1
    
    # Get the annotations. Since their times are in ticks, we need to convert them to seconds using the tempo map
    annotations=[]
    for annotation in jams_data.annotations:
        if annotation.namespace=='note_tab':
            base_tuning=annotation.sandbox["open_tuning"]
            data=annotation.data
            for note in data:
               
                
                #Convert from ticks to seconds
                ticks_per_beat=guitarpro.Duration.quarterTime
                note_begin_s=0
                note_end_s=0
                last_tick_jump=0
                # The mapping from ticks to seconds is a piecewise constant function on the tick axis, with discontinuities at the tempo change ticks.
                # integrate over tempos for the start time

                for tempo in tempos:
                    current_tempo=tempo[1]
                    seconds_per_tick=60/(current_tempo*ticks_per_beat)
                    if tempo[0]<note.time:
                        delta_ticks=tempo[0]-last_tick_jump
                        note_begin_s+=seconds_per_tick*delta_ticks
                        last_tick_jump=tempo[0]
                    
                delta_ticks=note.time-last_tick_jump
                note_begin_s+=seconds_per_tick*delta_ticks
                # integrate over tempos for the end time
                last_tick_jump=0
                noteend_ticks=note.time+note.duration
                for tempo in tempos:
                    current_tempo=tempo[1]
                    seconds_per_tick=60/(current_tempo*ticks_per_beat)
                    if tempo[0]<noteend_ticks:
                        delta_ticks=tempo[0]-last_tick_jump
                        note_end_s+=seconds_per_tick*delta_ticks
                        last_tick_jump=tempo[0]
                    
                delta_ticks=noteend_ticks-last_tick_jump
                note_end_s+=seconds_per_tick*delta_ticks
                        
                
                
                noteduration=note_end_s-note_begin_s

                annotations.append({
                    'time':note_begin_s,
                    'duration':noteduration,
                    'midi_note':note.value['fret']+base_tuning,
                    'confidence':note.confidence
                })
    return annotations,has_multiple_tempos
    # max_time=max(note['time']+note['duration'] for note in annotations)

    # num_bins=int(math.ceil(max_time/duration_bin_s))
    # binned_annotations=[[] for _ in range(num_bins)]
    # for note in annotations:
    #     bin_idx=int(note['time']//duration_bin_s)
    #     binned_annotations[bin_idx].append(note)
    # return binned_annotations,num_bins,max_time

def create_pianoroll(annotations,audio_mask,segment_start_s=None,segment_end_s=None, sr=48000, hop_length=256,  max_note=OUTPUT_DIM_NOTES,onsets_2d=False):
    max_time=max(note['time']+note['duration'] for note in annotations)
    # frame_time=1/sr
    res_start_sample=0
    if segment_start_s is not None and segment_end_s is not None:
        segment_start_sample,segment_end_sample=convert_period_s_to_samples(segment_start_s,segment_end_s,sr)
        res_start_sample=segment_start_sample

        num_samples=segment_end_sample-segment_start_sample
        num_samples=min(num_samples,len(audio_mask))
        num_notes=max_note

        piano_roll=np.zeros((num_notes,num_samples)) 
        
        # 1. Extract raw data into arrays
        times = np.array([n['time'] for n in annotations])
        durations = np.array([n['duration'] for n in annotations])
        ends = times + durations
        notes = np.array([n['midi_note'] for n in annotations], dtype=int)


        
        # 2. Vectorized Intersection Logic (The "if" replacement)
        # Instead of checking note by note, we find all valid notes in one go
        mask = (ends > segment_start_s) & (times < segment_end_s) & (notes >= 0) & (notes < num_notes)

        # 3. Apply mask and clip times to the segment boundaries
        # This handles the min/max logic in bulk
        v_starts = np.maximum(times[mask], segment_start_s)
        v_ends = np.minimum(ends[mask], segment_end_s)
        v_notes = notes[mask]
        # Latency compensation, get the delay of the quickest filter for each note. This happens to be the highest harmonic
        start_delays=np.array([get_butterworth_group_delay(n,num_harmonics,Q_FACTOR,sr) for n in v_notes],dtype=int  )

        # Latency compensation, get the delay of the slowest filter for each note. This happens to be the fundamental
        end_delays=np.array([get_butterworth_group_delay(n,1,Q_FACTOR,sr) for n in v_notes],dtype=int  )

        # 4. Convert to relative frame indices
        # (v_starts - segment_start_s) * sr
        relative_starts = ((v_starts - segment_start_s) * sr+start_delays).astype(int)
        relative_ends = ((v_ends - segment_start_s) * sr+end_delays).astype(int)

        # 5. Global Clipping (Safety check)
        relative_starts = np.clip(relative_starts, 0, num_samples)
        relative_ends = np.clip(relative_ends, 0, num_samples)

        # 6. Fill the piano roll
        # Note: Since you defined piano_roll as (num_frames, num_notes)
        for s, e, n in zip(relative_starts, relative_ends, v_notes):
            piano_roll[n,s:e] = 1#audio_mask[s:e] #Will mask labels at training time


    else:
        start_time=time.time()
        num_samples=int(math.ceil(max_time *sr))
        num_samples=min(num_samples,len(audio_mask))
        num_notes=max_note
        piano_roll=np.zeros((num_notes,num_samples))
            # def process_note(note):
            #     start_frame,end_frame=convert_period_s_to_frames(note['time'],note['time']+note['duration'],sr)
            #     note_idx=int(note['midi_note'])
            #     if note_idx>=0 and note_idx<num_notes:
            #         piano_roll[start_frame:end_frame,note_idx]=1

            # Parallel(n_jobs=1,backend='threading')(delayed(process_note)(note) for note in annotations)    
            # 1. Pre-calculate all indices at once (Vectorized)
        times = np.array([n['time'] for n in annotations])
        durations = np.array([n['duration'] for n in annotations])
        notes = np.array([n['midi_note'] for n in annotations], dtype=int)
        # Latency compensation, get the delay of the quickest filter for each note. This happens to be the highest harmonic
        start_delays=np.array([get_butterworth_group_delay(n,num_harmonics,Q_FACTOR,sr) for n in notes],dtype=int  )

        # Latency compensation, get the delay of the slowest filter for each note. This happens to be the fundamental
        end_delays=np.array([get_butterworth_group_delay(n,1,Q_FACTOR,sr) for n in notes],dtype=int  )

            # 2. Convert to frame indices in bulk
        start_frames = (times * sr+start_delays).astype(int)
        end_frames = ((times + durations) * sr+end_delays).astype(int)

            # 3. Clip frames to stay within array bounds
        start_frames = np.clip(start_frames, 0, num_samples - 1)
        end_frames = np.clip(end_frames, 0, num_samples)
        end_time=time.time()
        print(f"Time spent preparing indices: {end_time - start_time:.4f} seconds")

            # 4. Efficiently fill the piano roll
            # We use a simple loop here because slice-assignment is extremely fast in NumPy
        # Track the time spent in this loop
        start_time=time.time()
        for s, e, n_idx in zip(start_frames, end_frames, notes):
            if 0 <= n_idx < num_notes:
                e=min(num_samples,e)
                # print("Start:",s," End:",e)
                # print(piano_roll.shape)
                # print(audio_mask.shape)
                # print(piano_roll[n_idx,s:e].shape)
                # print(audio_mask[s:e].shape)
                piano_roll[n_idx,s:e] =1#audio_mask[s:e]   Will mask labels at training time
        end_time=time.time()
        
        print(f"Time spent filling piano roll: {end_time - start_time:.4f} seconds")  
    start_time=time.time()
    # Ensure at least one note is active per frame
    # 1. Find which frames have no notes active (max across note axis is 0)
    # This returns a boolean array of shape (num_frames,)
    empty_frames = np.where(np.max(piano_roll[:(OUTPUT_DIM_NOTES-1), :], axis=0) == 0)

    # 2. Fill the specific "silent" index for all those frames at once
    # If OUTPUT_DIM_NOTES-1 is your 'silence' or 'padding' note:
    piano_roll[OUTPUT_DIM_NOTES - 1, empty_frames] = 1

    end_time = time.time()
    print(f"Time spent ensuring at least one note per frame: {end_time - start_time:.4f} seconds")
    return piano_roll,res_start_sample

def load_jams_annotations(file_path, sr=48000, hop_length=256, max_note=OUTPUT_DIM_NOTES,onsets_2d=False):
    jams_data=load_jams_file(file_path)
    return extract_guitar_annotations(jams_data)


I0000 00:00:1775552025.277692   43954 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775552025.301361   43954 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775552025.816195   43954 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
from fretboardnonredundant import FretBoard
from common import frame_size,reshape_to_nn_input,reshape_to_nn_output,save_data_slices,plot_heatmap
from IPython.display import display
import os
from os import path
from scipy import io
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import glob 
import tensorflow as tf
import time
import random
#serialize example for a path object, an integer and numpy array
def _int64_feature(value: int) -> tf.train.Feature:
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def _path_feature(value: str) -> tf.train.Feature:
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value.encode()]))

def _bytes_feature(value: bytes) -> tf.train.Feature:
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def serialize_example(audio_path: str,frame_nr: int, output: np.ndarray) -> bytes:
    feature = {
        "audio_path":  _path_feature(audio_path),
        "frame_nr": _int64_feature(frame_nr),
        "output": _bytes_feature(output.tobytes()),
    }
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return example.SerializeToString()

def write_tfrecord(path, audio_path,start_frame, output):
    with tf.io.TFRecordWriter(path) as w:
        for frame in range(output.shape[0]):
            ex = serialize_example(audio_path,frame+start_frame, output[frame])
            w.write(ex)
# Files where converted with for file in *.wav; do sox -SG "$file" -r 48000 -e float -b 32 48k/"$file"; done
def process_annotation_segment(annotations,audio,audio_path,dirname,sample_rate,segment_start_s=None,segment_end_s=None):
        start_timing_segment=time.time()
        # Create output directories
        if segment_start_s is not None and segment_end_s is not None:
            segment_label=f"{int(segment_start_s)}s_to_{int(segment_end_s)}s"
            outdir=os.path.join(dirname,segment_label,'output')
            indir=os.path.join(dirname,segment_label,'input')
        else:
            outdir=os.path.join(dirname,'output')
            indir=os.path.join(dirname,'input')
        # return if both dirs exist
        if os.path.exists(outdir) and os.path.exists(indir):
            print(f"Data slices already exist for segment {segment_start_s} to {segment_end_s}, skipping")
            return False
        # Compute the piano roll and onsets
        audio_abs = np.abs(audio)

        frames =np.lib.stride_tricks.sliding_window_view(audio_abs, frame_size*2) # Evaluating 2*framesize=512 such that the latency of the  lowest butterworth at 82 is captured: Q/(pi*82)*sr=372 samples for Q=2 and sr=48000

        audio_mask_frames = np.where(np.max(frames, axis=1) > 0.05)
        audio_mask = np.zeros_like(audio)

        audio_mask[audio_mask_frames] = 1
        start_time=time.time()
        piano_roll,start_frame=create_pianoroll(annotations,audio_mask,segment_start_s=segment_start_s,segment_end_s=segment_end_s,sr=sample_rate,hop_length=frame_size,max_note=OUTPUT_DIM_NOTES,onsets_2d=False)
        start_frame=start_frame//frame_size
        frames=[]
        audio_mask=[]
        audio_mask_frames=[]
        audio_abs=[]
        # display(plot_heatmap(piano_roll))
        stop_time=time.time()
        print(f"Piano roll creation time: {stop_time-start_time:.2f} seconds")
        # We want to cut out sections around the onsets
        # Compute the positions to preserve
        start_time=time.time()
        # onset_positions=np.where(onsets>0)[0]
        # for onset in onset_positions:
        #     onsets[(onset-4*frame_size):onset+4*frame_size]=1

        # preserve_indices=np.where(onsets==0)[0]


        
        # Match piano roll length to audio length
        lenpianorool=piano_roll.shape[1]
        desired_length=len(audio)
        if lenpianorool<desired_length:
            # Pad with zeros
            padding = desired_length - lenpianorool
            piano_roll = np.pad(piano_roll, ((0, 0), (0, padding)), mode='constant')
        elif lenpianorool>desired_length:
            # Truncate
            piano_roll = piano_roll[:, :desired_length]
        end_time=time.time()
        print(f"Piano roll length adjustment time: {end_time-start_time:.2f} seconds")
        # # # Visualize the pruned data
        # filter =FretBoard(2,sample_rate)
        # numfilters=filter.get_num_filters()
        # filterbank_out=np.zeros((numfilters,len(audio)))
        # filter.process(audio,filterbank_out)
        # display(plot_heatmap(filterbank_out))
        # # display(plot_heatmap(piano_roll))
        # # # audio_plt=np.expand_dims(audio,axis=0)
        # # # display(plot_heatmap(audio_plt))
        # # display(plot_heatmap(filterbank_out))

        # # # Visualize the pruned data with string-oriented filters
        # filter =FretBoard(4,sample_rate)
        # numfilters=filter.get_num_filters()
        # filterbank_out=np.zeros((numfilters,len(audio)))
        # filter.process(audio,filterbank_out)
        # display(plot_heatmap(filterbank_out))

        # filter =FretBoard(6,sample_rate)
        # numfilters=filter.get_num_filters()
        # filterbank_out=np.zeros((numfilters,len(audio)))
        # filter.process(audio,filterbank_out)
        # display(plot_heatmap(filterbank_out))

        # filter =FretBoard(8,sample_rate)
        # numfilters=filter.get_num_filters()
        # filterbank_out=np.zeros((numfilters,len(audio)))
        # filter.process(audio,filterbank_out)
        # display(plot_heatmap(filterbank_out))
        # display(plot_heatmap(piano_roll))

        start_time=time.time()
        #Reshape audio to 2D (channels, frames)
        audio=np.expand_dims(audio,axis=0)
        print("Sizes after reshaping:"+str(audio.shape)+", "+str(piano_roll.shape))
        assert audio.shape[1]==piano_roll.shape[1],"Mismatched shapes"
        end_time=time.time()
        print(f"Onset pruning time: {end_time-start_time:.2f} seconds")

        # Reshape to NN input/output format
        start_time=time.time()
        # input_slices=reshape_to_nn_input(audio)
        

        # print(audio.shape)
        # print(audio_mask)
        # audio_mask=np.expand_dims(audio_mask,axis=0)
        # display(plot_heatmap(audio_mask[:,:2000],downsample_factor=1))
        # display(plot_heatmap(piano_roll[:,:2000],downsample_factor=1))
        # display(plot_heatmap(np.abs(audio[:,:2000]),downsample_factor=1))
        output_slices=reshape_to_nn_output(piano_roll)
        piano_roll=[]
        audio=[]
        end_time=time.time()
        print(f"Reshaping to NN input/output time: {end_time-start_time:.2f} seconds")

        
        
        Path(outdir).mkdir(parents=True,exist_ok=True)
        Path(indir).mkdir(parents=True,exist_ok=True)
        print("Not cutting out silence")
        # start_time=time.time()

        # silent_range_step=frame_size*4
        # ranges_to_preserve=np.zeros(output_slices.shape[0],dtype=bool)
        # for i in range(0,output_slices.shape[0],silent_range_step):
        #     end=min(i+silent_range_step,output_slices.shape[0])
        #     if np.sum(output_slices[i:end,(OUTPUT_DIM_NOTES-1)])==0:
        #         ranges_to_preserve[i:end]=True
        #         # print(f"Marking  range for preservation: {i} to {i+silent_range_step}")

        # percentage_preserved=100.0*np.sum(ranges_to_preserve)/ranges_to_preserve.shape[0]
        # print(f"Preserving {percentage_preserved:.2f}% of data slices after removing silent ranges")
        # end_time=time.time()
        # print(f"Silent range removal time: {end_time-start_time:.2f} seconds")

        # input_slices=(input_slices[ranges_to_preserve]).astype(np.float32)
        # output_slices=(output_slices[ranges_to_preserve]).astype(np.int8)
        # input_slices=(input_slices).astype(np.float32)
        output_slices=(output_slices).astype(np.int8)
        #create tensorflow dataset
        print("Creating tensorflow dataset")
        print("Audio path: ",audio_path)
        print("Start frame: ",start_frame)
        start_time=time.time()
        write_tfrecord(os.path.join(indir,'data.tfrecord'),audio_path,start_frame,output_slices)
        end_time=time.time()
        print(f"TFRecord writing time: {end_time-start_time:.2f} seconds")
        end_timing_segment=time.time()
        print(f"Total time for segment processing: {end_timing_segment-start_timing_segment:.2f} seconds")
        
        # dataset=tf.data.Dataset.from_tensor_slices((input_slices,output_slices))
        # # save the data slices
        # tf.data.Dataset.save(dataset,indir)
        # print("Saving input data to ",indir)
        
        # save_data_slices(indir,input_slices,chunk_size)
        
        # print("Saving output data to ",outdir)
        
        # save_data_slices(outdir,output_slices,chunk_size)
        return True

def prepare_audio_midi_data(jams_file):

    # Compute the piano roll and onsets
    print(f"Processing {jams_file}")
    
        #Output directory
    # basename=os.path.basename(jams_file).replace('.jams','')
    jamsdirname=os.path.dirname(jams_file)
    #get the parent directory
    jamsdirname=os.path.relpath(jamsdirname,start=jams_root_dir)
    dirname=os.path.join(output_data_dir,jamsdirname)
    #skip if output directory already exists

    if os.path.exists(dirname):
        print("Output directory already exists, skipping ",dirname)
        return False
    annotations,has_mult_temps=load_jams_annotations(jams_file)

    if has_mult_temps:
        print("Ignoring file with multiple tempos")
        return False



    
    #Input audio
    audio_file=jams_file.replace('annotations.jams','audio_48k.flac')#path.join(input_audio,basename+'_mix.wav')
    print("Loading audio file ",audio_file)
    if os.path.exists(audio_file)==False:
        print("Audio file does not exist, skipping ",audio_file)
        return False
    
    audio,sample_rate=librosa.load(audio_file,sr=48000)#sf.read(audio_file)



    # Normalize the audio
    maxaudio=np.max(np.abs(audio))
    audio=audio/maxaudio
    # plt.plot(audio)
    # plt.show()

    # Check the sample rate
    if sample_rate!=48000:
        print("Unexpected sample rate ",sample_rate)
        return False
    audiolen_s=len(audio)/sample_rate
    print(f"Audio length: {audiolen_s:.2f} seconds")
    segment_duration_s=2*60
    if audiolen_s>segment_duration_s:
        print("Processing in smaller segments")
        num_segments=int(math.ceil(audiolen_s/segment_duration_s))
        print(f"Number of segments: {num_segments}")
        res=False
        for seg_idx in range(num_segments):
            segment_start_s=seg_idx*segment_duration_s
            segment_end_s=min((seg_idx+1)*segment_duration_s,audiolen_s)
            print(f"Processing segment {seg_idx+1}/{num_segments}: {segment_start_s:.2f}s to {segment_end_s:.2f}s")
            # Extract the audio segment
            start_sample,end_sample=convert_period_s_to_samples(segment_start_s,segment_end_s,sample_rate)
            end_sample=min(end_sample,len(audio))
            audio_segment=audio[start_sample:end_sample]
            res=res|process_annotation_segment(annotations,audio_segment,audio_file,dirname,sample_rate,segment_start_s=segment_start_s,segment_end_s=segment_end_s)
        return res

    else:
        return process_annotation_segment(annotations,audio,audio_file,dirname,sample_rate)

    
import subprocess
import sys

def get_size_with_du(path):
    try:
        # -s: summary (total for the path)
        # -b: output in bytes
        result = subprocess.check_output(['du', '-sb', path], stderr=subprocess.DEVNULL)
        # result is like b'536870912000\t/path/to/dir\n'
        return int(result.split()[0])
    except Exception as e:
        print(f"Error checking directory size: {e}")
        return 0

from joblib import Parallel, delayed

# Flatten files as shown above
all_jams_files = glob.glob(os.path.join(jams_root_dir,'**', '*.jams'),recursive=True)# [f for d in jams_dirs for f in sorted(glob.glob(os.path.join(d,'**', '*.jams'),recursive=True))]

#Parallel(n_jobs=5)(delayed(prepare_audio_midi_data)(f) for f in all_jams_files) 

random.seed(42)
random.shuffle(all_jams_files)

                
start_time=time.time()
files_processed=0
# Problematic jams files with multiple tempos
# all_jams_files=['/data/converted_audio_jams/clean/stratocaster_clean_bridge/System Of A Down - Toxicity (2) - gp3__1 - Electric Clean Guitar__midi/annotations.jams',
#                 '/data/converted_audio_jams/distorted/lespaul_clean_neck/Composers Of MSB - Until It Die\'s - gp4__2 - Distortion Guitar__midi/annotations.jams',
#                 '/data/converted_audio_jams/clean/stratocaster_clean_bridge/Alpha Current - The Lottery, Part I - gp4__1 - Electric Clean Guitar__midi/annotations.jams',]

# all_jams_files=['/data/converted_audio_jams/clean/lespaul_clean_bridge/Al Green - Lets Stay Together - gp3__2 - Electric Clean Guitar__midi/annotations.jams',
#                 '/data/converted_audio_jams/clean/lespaul_clean_bridge/Jazz Licks - Bass Lines On guitar - gp3__1 - Electric Jazz Guitar__midi/annotations.jams',
#                 '/data/converted_audio_jams/clean/lespaul_clean_bridge/Nirvana - Spank Thru - gp4__1 - Electric Clean Guitar__midi/annotations.jams',
#                 '/data/converted_audio_jams/clean/lespaul_clean_bridge/Soundgarden - Spoonman - gp3__3 - Electric Clean Guitar__midi/annotations.jams',
#                 '/data/converted_audio_jams/clean/stratocaster_clean_bridge/Eyesburn - No Free Time (Solo) - gp3__2 - Electric Clean Guitar__midi/annotations.jams',
#                 '/data/converted_audio_jams/clean/stratocaster_clean_bridge/Metallica - Wasting My Hate - gp3__4 - Electric Clean Guitar__midi/annotations.jams',
#                 '/data/converted_audio_jams/clean/stratocaster_clean_bridge/Weezer - El Scorcho - gp4__5 - Electric Clean Guitar__midi/annotations.jams',
#                 '/data/converted_audio_jams/distorted/stratocaster_clean_neck/John Death - Alter Of Blood - gp4__2 - Distortion Guitar__midi/annotations.jams']
for jams_file in all_jams_files:
    # process the file, measuring the time and estimating the time remaining based on the average time per file and the number of files left
    
    if prepare_audio_midi_data(jams_file):
        current_time=time.time()
        elapsed_time=current_time-start_time
        files_processed=files_processed+1
        avg_time_per_file=elapsed_time/files_processed
        files_left=len(all_jams_files)-files_processed
        est_time_remaining=avg_time_per_file*files_left
        print(f"Estimated time remaining: {est_time_remaining/60:.2f} minutes")
        current_size = get_size_with_du(output_data_dir)
        print(f"Current dataset size: {current_size / (1024**3):.2f} GB")
        if current_size >= limit_bytes:
            print(f"Reached size limit of {limit_gb} GB, stopping data preparation.")
            break

    
            

    


Processing /data2/converted_audio_jams/luthier_pick/Anderson, Muriel - Home Town - gp4__1 - Acoustic Nylon Guitar__midi/annotations.jams
Output directory already exists, skipping  /data2/data_slices/luthier_pick/Anderson, Muriel - Home Town - gp4__1 - Acoustic Nylon Guitar__midi
Processing /data2/converted_audio_jams/luthier_pick/Albeniz, Isaac - C_rdoba - gp4__1 - Acoustic Nylon Guitar__midi/annotations.jams
Ignoring file with multiple tempos
Processing /data2/converted_audio_jams/luthier_pick/Bach, Johann Sebastian - prelude in D minor - gp3__1 - Acoustic Nylon Guitar__midi/annotations.jams
Output directory already exists, skipping  /data2/data_slices/luthier_pick/Bach, Johann Sebastian - prelude in D minor - gp3__1 - Acoustic Nylon Guitar__midi
Processing /data2/converted_audio_jams/luthier_pick/Classical Competition 2 -  Queen Of The Sea - gp4__1 - Acoustic Steel Guitar__midi/annotations.jams
Ignoring file with multiple tempos
Processing /data2/converted_audio_jams/taylor_finger/Ba

In [4]:
print("End block executed")

End block executed
